In [ ]:
import os
import pyspark

# Autodetekcja wersji Sparka → właściwy connector
spark_version = pyspark.__version__
print(f"Wykryta wersja PySpark: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    # Spark 3.x
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Użyty connector:        {KAFKA_PACKAGE}")

# Musi być ustawione PRZED SparkSession.builder
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell'

In [ ]:
from pyspark.sql import SparkSession

KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
) #biblioteka którą ściagam za każdym razem gdy tworzę kontekst sparka
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
df = kafka_raw.select("value")
 
query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

In [3]:
%%file test.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
df = kafka_raw.select("value")
 
query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    #.option("truncate", False) 
    .start()
)
 
query.awaitTermination()

Overwriting test.py


In [ ]:
#postac bitowa binarna którą kafka wysyła

In [2]:
#faza 1 - cast string
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
df = kafka_raw.select(col("value").cast("string"))
 
query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+-----+
|value|
+-----+
+-----+

Batch ID: 1
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                            |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"tx_id": "TX4697", "user_id": "u15", "amount": 2710.91, "store": "Krak\u00f3w", "category": "\u017cywno\u015b\u0107", "timestamp": "2026-04-27T10:43:56.493325"}|
|{"tx_id": "TX7160", "user_id": "u19", "amount": 662.25, "store": "Krak\u00f3w", "category": "ksi\u0105\u017cki", "timestamp": "2026-04-27T10:43:57.494055"}      |
|{"tx_id": "TX2736", "user_id": "u05", "amount": 4378.55, "store": "Krak\u00f3w", "category": "elektronika", "timestamp": "

In [1]:
#faza 2 - json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
 
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
df = ( kafka_raw.select(
        from_json(col("value").cast("string"), tx_schema).alias("tx")
     )
)
 
query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+---+
|tx |
+---+
+---+

Batch ID: 1
+------------------------------------------------------------------------+
|tx                                                                      |
+------------------------------------------------------------------------+
|{TX7519, u18, 4117.2, Warszawa, elektronika, 2026-04-27T10:45:16.593144}|
+------------------------------------------------------------------------+

Batch ID: 2
+-------------------------------------------------------------------+
|tx                                                                 |
+-------------------------------------------------------------------+
|{TX9822, u11, 2893.87, Gdańsk, odzież, 2026-04-27T10:45:17.593476} |
|{TX1038, u12, 1990.23, Gdańsk, żywność, 2026-04-27T10:45:18.594624}|
+-------------------------------------------------------------------+

Batch ID: 3
+--------------------------------------------------------------------+
|tx                                                        

In [1]:
#faza 3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
 
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
df = ( kafka_raw.select(
        from_json(col("value").cast("string"), tx_schema).alias("tx")
     )
)
 
df2 = ( df.select("tx.*")
      .withColumn("timestamp", to_timestamp("timestamp"))
      )
 
query = (df2.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+-----+-------+------+-----+--------+---------+
|tx_id|user_id|amount|store|category|timestamp|
+-----+-------+------+-----+--------+---------+
+-----+-------+------+-----+--------+---------+

Batch ID: 1
+------+-------+-------+-------+--------+--------------------------+
|tx_id |user_id|amount |store  |category|timestamp                 |
+------+-------+-------+-------+--------+--------------------------+
|TX9233|u01    |1696.69|Wrocław|odzież  |2026-04-27 10:55:17.372441|
|TX8909|u11    |12.44  |Wrocław|żywność |2026-04-27 10:55:18.373097|
|TX3561|u01    |1921.38|Wrocław|książki |2026-04-27 10:55:19.374914|
+------+-------+-------+-------+--------+--------------------------+

Batch ID: 2
+------+-------+-------+-------+--------+--------------------------+
|tx_id |user_id|amount |store  |category|timestamp                 |
+------+-------+-------+-------+--------+--------------------------+
|TX7750|u17    |1037.27|Kraków |odzież  |2026-04-27 10:55:20.379387|
|TX1211|u04

In [2]:
#parsed version
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
 
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

parsed = (
    kafka_raw.select(from_json(col("value").cast("string"), tx_schema).alias("tx")
    ).select("tx.*").withColumn("timestamp", to_timestamp("timestamp"))
)
 
query = (parsed.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+-----+-------+------+-----+--------+---------+
|tx_id|user_id|amount|store|category|timestamp|
+-----+-------+------+-----+--------+---------+
+-----+-------+------+-----+--------+---------+

Batch ID: 1
+------+-------+-------+-------+-----------+--------------------------+
|tx_id |user_id|amount |store  |category   |timestamp                 |
+------+-------+-------+-------+-----------+--------------------------+
|TX3498|u16    |3943.82|Gdańsk |książki    |2026-04-27 10:58:08.556133|
|TX2417|u06    |3937.41|Wrocław|żywność    |2026-04-27 10:58:09.557581|
|TX5438|u16    |2827.96|Kraków |elektronika|2026-04-27 10:58:10.560677|
+------+-------+-------+-------+-----------+--------------------------+

Batch ID: 2
+------+-------+-------+--------+-----------+--------------------------+
|tx_id |user_id|amount |store   |category   |timestamp                 |
+------+-------+-------+--------+-----------+--------------------------+
|TX7631|u19    |4923.89|Warszawa|elektronika|20

In [1]:
#okno 1minutowe per sklep
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp, window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import window, count, sum as _sum, round as _round

 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
def process_batch(df, batch_id, tstop=40):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

parsed = (
    kafka_raw.select(from_json(col("value").cast("string"), tx_schema).alias("tx")
    ).select("tx.*").withColumn("timestamp", to_timestamp("timestamp"))
)

windows = (parsed.withWatermark("timestamp", "2 seconds")
           .groupBy(window("timestamp", "10 seconds"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    ))

query = (windows.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+------+-----+---------+--------+
|window|store|liczba_tx|suma_PLN|
+------+-----+---------+--------+
+------+-----+---------+--------+

Batch ID: 1
+------+-----+---------+--------+
|window|store|liczba_tx|suma_PLN|
+------+-----+---------+--------+
+------+-----+---------+--------+

Batch ID: 2
+------------------------------------------+--------+---------+--------+
|window                                    |store   |liczba_tx|suma_PLN|
+------------------------------------------+--------+---------+--------+
|{2026-04-27 11:12:20, 2026-04-27 11:12:30}|Warszawa|2        |1509.86 |
+------------------------------------------+--------+---------+--------+

Batch ID: 3
+------------------------------------------+--------+---------+--------+
|window                                    |store   |liczba_tx|suma_PLN|
+------------------------------------------+--------+---------+--------+
|{2026-04-27 11:12:30, 2026-04-27 11:12:40}|Kraków  |3        |8273.32 |
|{2026-04-27 11:12:3

In [ ]:
#do domu
#zad 4.2
#czesc 5 jakies alerty wysylamy do kafki?